# Notebook de KPIs financeiros
Este notebook calcula e valida os principais indicadores financeiros do projeto.

A análise será realizada por cenário e por ano.

In [1]:
import pandas as pd
import numpy as np

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

caminho = "../staging/dados_tratados.csv"

df = pd.read_csv(
    caminho,
    sep=";",
    encoding="utf-8-sig"
)

print("Dimensões da base:", df.shape)
print("Quantidade de cenários:", df["Cenario"].nunique())
print("Quantidade de anos:", df["Ano"].nunique())
print("Quantidade de contas:", df["Conta"].nunique())

display(df.head(10))

Dimensões da base: (958800, 4)
Quantidade de cenários: 1200
Quantidade de anos: 12
Quantidade de contas: 69


,Cenario,Ano,Conta,Valor_numerico
0,Total Cen_00001,Ano 1,BAL - Amortização - Intangível,"-5.047.123.846,36"
1,Total Cen_00001,Ano 1,BAL - Amortização Acumulada,"-30.111.275,81"
2,Total Cen_00001,Ano 1,BAL - At Fiscal Diferido,"-103.969.289,50"
3,Total Cen_00001,Ano 1,BAL - Ativo Circulante,"1.795.720.872,18"
4,Total Cen_00001,Ano 1,BAL - Capital Social,"-309.618.939,00"
5,Total Cen_00001,Ano 1,BAL - Contas a Pagar - Parte Relacionada,"-14.086.270,70"
6,Total Cen_00001,Ano 1,BAL - Contas a Receber - Clientes,"453.870.957,15"
7,Total Cen_00001,Ano 1,BAL - Contas a Receber - Partes Relacionadas,"584.661,47"
8,Total Cen_00001,Ano 1,BAL - Contas a Receber - SWAP,"95.680.421,92"
9,Total Cen_00001,Ano 1,BAL - Créditos Tributários,"1.846.452,68"


## Organização das contas

Para calcular os indicadores, vamos organizar a base para que cada combinação de cenário e ano tenha as contas como colunas.

Assim, cada linha representará um cenário em determinado ano.

In [2]:
base_indicadores = (
    df
    .pivot_table(
        index=["Cenario", "Ano"],
        columns="Conta",
        values="Valor_numerico",
        aggfunc="first"
    )
    .reset_index()
)

print("Dimensões da base para os indicadores:", base_indicadores.shape)

display(base_indicadores.head())

Dimensões da base para os indicadores: (14400, 71)


Conta,Cenario,Ano,BAL - Amortização - Intangível,BAL - Amortização Acumulada,BAL - At Fiscal Diferido,BAL - Ativo Circulante,BAL - Capital Social,BAL - Contas a Pagar - Parte Relacionada,BAL - Contas a Receber - Clientes,BAL - Contas a Receber - Partes Relacionadas,...,FLU - Distribuição para Acionista,FLU - Entradas,FLU - Geração de Caixa,FLU - Imposto de Renda e Contribuição Social,FLU - Investimentos,FLU - Receita,FLU - Resultado Financeiro,FLU - Saldo Final,FLU - Saldo Inicial,FLU - Tributos
0,Total Cen_00001,Ano 1,"-5.047.123.846,36","-30.111.275,81","-103.969.289,50","1.795.720.872,18","-309.618.939,00","-14.086.270,70","453.870.957,15","584.661,47",...,"-1.532.407.884,49","91.626.997,28","430.649.781,90","-934.472.751,30","-534.001.210,70","5.396.830.757,26","-730.404.384,16","1.205.264.965,52","774.615.183,61","-473.257.578,14"
1,Total Cen_00001,Ano 10,"-12.000.284.706,94","-223.407.545,04","1.688.778.561,98","2.461.671.910,58","-309.618.939,00","-14.332.141,74","405.567.948,90","584.661,47",...,"-3.798.611.798,70","199.176.238,67","402.044.964,32","-2.101.015.948,61","-54.102.449,66","9.313.008.709,52","-137.375.079,77","2.325.378.098,86","1.923.333.134,54","-811.404.288,10"
2,Total Cen_00001,Ano 11,"-12.843.837.315,22","-274.644.262,99","1.859.161.220,86","3.209.590.420,02","-309.618.939,00","-14.332.141,74","405.567.948,90","584.661,47",...,"-4.256.469.927,16","226.483.810,97","920.431.807,91","-2.621.084.742,77","-42.678.751,87","9.918.855.773,33","-87.818.904,32","3.245.809.906,77","2.325.378.098,86","-863.933.596,64"
3,Total Cen_00001,Ano 12,"-0,00",NaN,"0,00","387.716.734,20",NaN,NaN,"-0,00",NaN,...,"-10.361.817.487,69","316.130.696,22","-2.827.614.976,45","-3.840.487.186,14","-7.674.905,61","13.454.776.849,17","22.051.476,85","418.194.930,32","3.245.809.906,77","-1.064.907.832,16"
4,Total Cen_00001,Ano 2,"-5.710.883.589,35","-40.545.379,41","163.668.039,78","1.567.978.527,75","-309.618.939,00","-14.332.141,74","405.567.948,90","584.661,47",...,"-2.553.745.841,39","151.578.958,84","-182.837.520,74","-1.333.357.245,35","-332.034.978,46","5.872.797.811,43","-545.715.550,98","1.022.427.444,78","1.205.264.965,52","-505.006.210,74"


## Contas disponíveis para os indicadores

Antes de calcular os indicadores financeiros, verificamos quais contas estão disponíveis na base tratada.

In [3]:
contas_disponiveis = sorted(
    df["Conta"].dropna().unique()
)

print("Quantidade de contas disponíveis:", len(contas_disponiveis))

for conta in contas_disponiveis:
    print(conta)

Quantidade de contas disponíveis: 69
BAL - Amortização - Intangível
BAL - Amortização Acumulada
BAL - At Fiscal Diferido
BAL - Ativo Circulante
BAL - Capital Social
BAL - Contas a Pagar - Parte Relacionada
BAL - Contas a Receber - Clientes
BAL - Contas a Receber - Partes Relacionadas
BAL - Contas a Receber - SWAP
BAL - Créditos Tributários
BAL - Depreciação Acumulada
BAL - Diferido
BAL - Disponível
BAL - Dividendos Antecipados
BAL - Emprést
BAL - Empréstimos
BAL - Encargos Sociais e Trabalhistas
BAL - Estoques Diversos
BAL - Exigível a Longo Prazo
BAL - Fornecedores
BAL - Impostos
BAL - Investimentos - Imobilizado
BAL - Investimentos - Intangível
BAL - Obrigações com o Poder Concedente
BAL - Outorga da Concessão
BAL - Outros Créditos
BAL - Outros Créditos LP
BAL - Outros Débitos
BAL - Outros deb
BAL - Passivo Circulante
BAL - Patrimônio Líquido
BAL - Permanente
BAL - Prov para Contingências
BAL - Provisão Manutenção
BAL - Realizável a Longo Prazo
BAL - Reserva de Retenção de Lucros
BAL

## Contas utilizadas nos indicadores

A base possui contas já calculadas nos demonstrativos contábeis.

Nesta etapa, vamos identificar as principais contas da DRE e do Fluxo de Caixa que poderão ser utilizadas nos indicadores e nas análises de cenário.

In [4]:
contas_dre = sorted(
    df.loc[df["Conta"].str.startswith("DRE -"), "Conta"].unique()
)

contas_flu = sorted(
    df.loc[df["Conta"].str.startswith("FLU -"), "Conta"].unique()
)

print("CONTAS DA DRE:")
for conta in contas_dre:
    print(conta)

print("\nCONTAS DO FLUXO DE CAIXA:")
for conta in contas_flu:
    print(conta)

CONTAS DA DRE:
DRE - Custos
DRE - Depreciação e Amortização
DRE - Despesas Financeiras
DRE - EBITDA
DRE - Imposto de Renda e Contribuição Social
DRE - Outros Resultados Operacionais
DRE - Receita
DRE - Receitas Financeiras
DRE - Resultado Antes do Imposto de Renda
DRE - Resultado Financeiro
DRE - Resultado Líquido
DRE - Resultado Líquido após Equivalência
DRE - Resultado Operacional
DRE - Tributos

CONTAS DO FLUXO DE CAIXA:
FLU - Custos
FLU - Despesas Financeiras
FLU - Distribuição para Acionista
FLU - Entradas
FLU - Geração de Caixa
FLU - Imposto de Renda e Contribuição Social
FLU - Investimentos
FLU - Receita
FLU - Resultado Financeiro
FLU - Saldo Final
FLU - Saldo Inicial
FLU - Tributos


## Contas utilizadas do Balanço Patrimonial

In [5]:
contas_bal = sorted(
    df.loc[df["Conta"].str.startswith("BAL -"), "Conta"].unique()
)

print("CONTAS DO BALANÇO PATRIMONIAL:")
for conta in contas_bal:
    print(conta)

print("\nQuantidade de contas do BAL:", len(contas_bal))

CONTAS DO BALANÇO PATRIMONIAL:
BAL - Amortização - Intangível
BAL - Amortização Acumulada
BAL - At Fiscal Diferido
BAL - Ativo Circulante
BAL - Capital Social
BAL - Contas a Pagar - Parte Relacionada
BAL - Contas a Receber - Clientes
BAL - Contas a Receber - Partes Relacionadas
BAL - Contas a Receber - SWAP
BAL - Créditos Tributários
BAL - Depreciação Acumulada
BAL - Diferido
BAL - Disponível
BAL - Dividendos Antecipados
BAL - Emprést
BAL - Empréstimos
BAL - Encargos Sociais e Trabalhistas
BAL - Estoques Diversos
BAL - Exigível a Longo Prazo
BAL - Fornecedores
BAL - Impostos
BAL - Investimentos - Imobilizado
BAL - Investimentos - Intangível
BAL - Obrigações com o Poder Concedente
BAL - Outorga da Concessão
BAL - Outros Créditos
BAL - Outros Créditos LP
BAL - Outros Débitos
BAL - Outros deb
BAL - Passivo Circulante
BAL - Patrimônio Líquido
BAL - Permanente
BAL - Prov para Contingências
BAL - Provisão Manutenção
BAL - Realizável a Longo Prazo
BAL - Reserva de Retenção de Lucros
BAL - Res